# Fraud Detection — Silver Layer (PySpark)
**Bronze → Silver:** تنظيف البيانات، تصحيح الأنواع، معالجة القيم الناقصة

## 1. تشغيل SparkSession

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, DateType

import os

SPARK_MASTER = os.environ.get('SPARK_MASTER', 'local[*]')

spark = (
    SparkSession.builder
    .appName('fraud_detection')
    .master(SPARK_MASTER)
    .getOrCreate()
)

print("Spark version:", spark.version)

## 2. قراءة Bronze Layer (ملفات Parquet)

In [ ]:
# ── عدّل المسارات دي حسب مكان ملفاتك في Docker ──
BASE_PATH = "/app/ingestion/bronze"   # mounted project in Docker at /app

transactions_raw = spark.read.parquet(f"{BASE_PATH}/transactions_data")
cards_raw        = spark.read.parquet(f"{BASE_PATH}/cards_data")
users_raw        = spark.read.parquet(f"{BASE_PATH}/users_data")
mcc_raw          = spark.read.parquet(f"{BASE_PATH}/mcc_codes")
labels_raw       = spark.read.parquet(f"{BASE_PATH}/train_fraud_labels")

print("Transactions:", transactions_raw.count(), transactions_raw.columns)
print("Cards:       ", cards_raw.count(),        cards_raw.columns)
print("Users:       ", users_raw.count(),         users_raw.columns)

## 3. Silver — Transactions

In [ ]:
# 3‑A: تصحيح الأنواع
transactions = transactions_raw \
    .withColumn("date",   F.to_timestamp("date")) \
    .withColumn("amount", F.regexp_replace("amount", r"\$", "").cast(DoubleType()))

# 3‑B: merchant_state — القيم الناقصة لأسباب منطقية (ONLINE)
transactions = transactions.withColumn(
    "merchant_state",
    F.when(
        F.col("merchant_state").isNull() & (F.col("merchant_city") == "ONLINE"),
        F.lit("ONLINE")
    ).otherwise(F.col("merchant_state"))
)

# 3‑C: errors — ملء الناقص بـ 'No Errors'
transactions = transactions.withColumn(
    "errors",
    F.coalesce(F.col("errors"), F.lit("No Errors"))
)

# 3‑D: zip — ONLINE يأخذ '0'، باقي الدول من الـ mapping
missing_zip = {
    "Puerto Vallarta": "48300", "Vatican City": "00120", "Guadalajara": "44100",
    "Santo Domingo": "10101",  "Montreal": "H3A",       "Toronto": "M5H",
    "San Jose": "10101",       "Berlin": "10115",        "Mexico City": "01000",
    "Shanghai": "200000",      "Cancun": "77500",        "Edinburgh": "EH1",
    "Tallinn": "10111",        "Tokyo": "100-0001",      "Paris": "75001",
    "London": "SW1A 1AA",      "Amsterdam": "1011",      "Rome": "00184",
    "Madrid": "28001",         "Barcelona": "08001",     "Cairo": "11511",
    "Dubai": "00000",          "Abu Dhabi": "00000",     "Riyadh": "12611",
    "Moscow": "101000",        "Beijing": "100000",      "Mumbai": "400001",
    "Sydney": "2000",          "Singapore": "018989",    "Seoul": "04524",
    "Bangkok": "10200",        "Jakarta": "10110",       "Kuala Lumpur": "50000",
    "Istanbul": "34000",       "Warsaw": "00-001",       "Budapest": "1051",
    "Prague": "110 00",        "Vienna": "1010",         "Brussels": "1000",
    "Lisbon": "1100-148",      "Athens": "10552",        "Helsinki": "00100",
    "Stockholm": "11120",      "Oslo": "0150",           "Copenhagen": "1050",
    "Zurich": "8001",          "Geneva": "1201",         "Hong Kong": "999077",
    "Bogota": "110111",        "Lima": "15001",          "Santiago": "8320000",
    "Buenos Aires": "C1000",   "Sao Paulo": "01000-000", "Rio de Janeiro": "20000-000",
    "Montevideo": "11000",     "Panama City": "0819",    "Quito": "170101",
    "Nairobi": "00100",        "Addis Ababa": "1000",    "Rabat": "10000",
    "Algiers": "16000",        "Tunis": "1000",          "Hanoi": "100000",
    "Karachi": "74000",        "Lahore": "54000",        "Islamabad": "44000",
    "Dhaka": "1000",           "Colombo": "00100",       "Doha": "00000",
    "Muscat": "113",           "Beirut": "1107",         "Amman": "11118",
    "Tehran": "11369",         "Baghdad": "10001",       "Tashkent": "100000",
    "Reykjavik": "101",        "Dublin": "D01",          "Nicosia": "1010",
    "Wellington": "6011",      "Male": "20026",          "Majuro": "96960",
    "Tegucigalpa": "11101"
}

# بناء CASE WHEN expression من الـ dict
zip_map_expr = F.when(F.col("merchant_city") == "ONLINE", F.lit("0"))
for city, z in missing_zip.items():
    zip_map_expr = zip_map_expr.when(F.col("merchant_city") == city, F.lit(z))
zip_map_expr = zip_map_expr.otherwise(F.col("zip"))

transactions = transactions.withColumn(
    "zip",
    F.when(F.col("zip").isNull(), zip_map_expr).otherwise(F.col("zip"))
)
# ما تبقى من null يأخذ قيمة fallback
transactions = transactions.withColumn(
    "zip",
    F.coalesce(F.col("zip"), F.lit("01000-000"))
)

print("Nulls after cleaning:")
transactions.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in transactions.columns]).show()
transactions.printSchema()

## 4. Silver — Users

In [ ]:
users = users_raw \
    .withColumn("per_capita_income", F.regexp_replace("per_capita_income", r"\$", "").cast(DoubleType())) \
    .withColumn("yearly_income",     F.regexp_replace("yearly_income",     r"\$", "").cast(DoubleType())) \
    .withColumn("total_debt",        F.regexp_replace("total_debt",        r"\$", "").cast(DoubleType()))

# عمود مشتق: الوقت المتبقي للتقاعد
users = users.withColumn(
    "time_left_until_retirement",
    F.when(
        F.col("current_age") >= F.col("retirement_age"),
        F.lit("retired")
    ).otherwise(
        (F.col("retirement_age") - F.col("current_age")).cast("string")
    )
)

print("Users nulls:")
users.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in users.columns]).show()
users.printSchema()

## 5. Silver — Cards

In [ ]:
cards = cards_raw \
    .withColumn("credit_limit",  F.regexp_replace("credit_limit", r"\$", "").cast(DoubleType())) \
    .withColumn("expires",       F.to_date("expires",       "MM/yyyy")) \
    .withColumn("acct_open_date",F.to_date("acct_open_date","MM/yyyy"))

# عمود مشتق: مدة الحساب بالأيام
cards = cards.withColumn(
    "account_duration_days",
    F.datediff(F.col("expires"), F.col("acct_open_date"))
)

print("Cards nulls:")
cards.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in cards.columns]).show()
cards.printSchema()

## 6. Silver — MCC Codes

In [ ]:
# mcc_codes موجودة بالفعل كـ Parquet في Bronze layer
mcc_codes = mcc_raw

mcc_codes.show(5)
print("MCC rows:", mcc_codes.count())

## 7. Silver — Fraud Labels

In [ ]:
# train_fraud_labels موجودة كـ Parquet في Bronze layer
train_fraud_labels = labels_raw

train_fraud_labels.groupBy("target").count().show()
train_fraud_labels.printSchema()

## 8. Merges (Silver Joins)

In [ ]:
# 8‑A: Cards + Users
cards_users = cards.join(users, cards["client_id"] == users["id"], how="inner")

# 8‑B: Cards + Transactions
cards_transactions = cards.join(
    transactions,
    on=[(cards["id"] == transactions["card_id"]) & (cards["client_id"] == transactions["client_id"])],
    how="inner"
)

# 8‑C: Transactions + MCC
transactions_mcc = transactions.join(
    mcc_codes,
    transactions["mcc"].cast("string") == mcc_codes["mcc_code"],
    how="left"
)

# 8‑D: Transactions + Labels
transactions_labels = transactions.join(train_fraud_labels, on="id", how="left")

print("cards_users:        ", cards_users.count())
print("cards_transactions: ", cards_transactions.count())
print("transactions_mcc:   ", transactions_mcc.count())
print("transactions_labels:", transactions_labels.count())

## 9. كتابة Silver Layer (Parquet)

In [ ]:
SILVER_PATH = "/app/ingestion/silver"

transactions.write.mode("overwrite").parquet(f"{SILVER_PATH}/transactions")
users.write.mode("overwrite").parquet(f"{SILVER_PATH}/users")
cards.write.mode("overwrite").parquet(f"{SILVER_PATH}/cards")
mcc_codes.write.mode("overwrite").parquet(f"{SILVER_PATH}/mcc_codes")
train_fraud_labels.write.mode("overwrite").parquet(f"{SILVER_PATH}/fraud_labels")
transactions_labels.write.mode("overwrite").parquet(f"{SILVER_PATH}/transactions_with_labels")

print("✅ Silver Layer written successfully", SILVER_PATH)

In [ ]:
spark.stop()